In [ ]:
import pandas as pd
import yfinance as yf
from pyprojroot import here

# Processing Data

In [ ]:
def filter_dates(df):
    df = df[df['datetime'] >= '2019-01-01']
    df = df[df['datetime'] <= '2023-12-31']

    df = df.reset_index(drop=True)
    
    return df

## UK Covid-19 Cases

In [ ]:
df = pd.read_csv(here("data/raw/ukhsa-chart-download.csv"))

df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d")

df = df[["date", "metric_value"]].rename(
    columns={"date": "datetime", "metric_value": "cases"}
)

df = df.set_index("datetime").resample("MS").sum().reset_index()

df = filter_dates(df)

df.head()

In [ ]:
merged = df.copy()

## LHR Passengers

In [ ]:
df = pd.read_excel(here("data/raw/Jan_2026_Monthly_Traffic_Statistics.xlsx"), skiprows=2)

df["Month"] = pd.to_datetime(df["Month"], format="%Y-%m-%d")

df = df[["Month", "Passengers"]].rename(
    columns={"Month": "datetime", "Passengers": "passengers"}
)

df = df.set_index("datetime").resample("MS").sum().reset_index()

df = filter_dates(df)

df.head()

In [ ]:
merged = merged.merge(df, on="datetime", how="inner")

## Stock Price

In [ ]:
ticker = "PYPL"

df = yf.download(
    ticker, 
    start="2019-01-01", 
    end="2023-12-31",
    interval="1d"
)

df = df["Close"].reset_index()

df = df.rename(columns={"Date": "datetime", "Close": "stock_price"})

df = df.set_index("datetime").resample("MS").mean().reset_index().round(2)

df = filter_dates(df)

df.head()

In [ ]:
merged = merged.merge(df, on="datetime", how="inner")

In [ ]:
merged.head()

In [ ]:
merged.to_csv(here("data/processed/merged_data.csv"), index=False)